In [1]:
import torch
import torch.nn as nn

class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 特征提取部分
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=2)  # 1→6通道
        self.pool  = nn.MaxPool2d(kernel_size=2)                                          # 尺寸减半
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)            # 6→16通道
        self.relu  = nn.ReLU()
        # 分类部分（全连接）
        self.fc1 = nn.Linear(16 * 5 * 5, 120)   # 展平后接全连接
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)             # 输出 10 类

    def forward(self, x):
        # x: [batch, 1, 28, 28]
        x = self.pool(self.relu(self.conv1(x)))  # 卷积→激活→池化 → [batch,6,14,14]
        x = self.pool(self.relu(self.conv2(x)))  # 卷积→激活→池化 → [batch,16,5,5]
        x = x.view(x.size(0), -1)                # 展平 → [batch, 16*5*5]
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)                          # → [batch, 10]
        return x

model = LeNet()
print(model)

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (relu): ReLU()
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [3]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        images = images.to(device)          # 注意：不展平！CNN 直接吃 [b,1,28,28]
        labels = labels.to(device)
        outputs = model(images)             # 五步循环，和 Day 2 完全一样
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, loss={loss.item():.4f}")

# 评估（和 Day 2 一样，也不展平）
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100*correct/total:.2f}%")

Epoch 1, loss=0.0193
Epoch 2, loss=0.0643
Epoch 3, loss=0.0378
Epoch 4, loss=0.0035
Epoch 5, loss=0.0116
Test Accuracy: 98.77%
